In [1]:
import tensorflow_datasets as tfds

2023-10-23 04:49:25.799232: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/polivares/anaconda3/envs/DataScience/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset, info = tfds.load("tf_flowers", as_supervised=True, with_info=True)

2023-10-23 04:49:32.694666: I tensorflow/core/common_runtime/process_util.cc:146] Creating new thread pool with default inter op setting: 2. Tune using inter_op_parallelism_threads for best performance.


In [3]:
test_set, valid_set, train_set = tfds.load("tf_flowers",split=["train[0%:10%]", "train[10%:25%]", "train[25%:]"], as_supervised=True )

In [4]:
import tensorflow as tf

In [5]:
# Previo a la utilización de nuestro dataset en la red neuronal, la red que utilizaremos provee de
# funcionalidades de pre-procesamiento ya incluídas. Por lo tanto, utilizaremos estas funcionalidades
# y las dejaremos dentro de una función
def preprocess(image, label):
    # Cambiaremos las dimensiones de la imagen de entrada
    resized_image = tf.image.resize(image, [224, 224]) # Guardamos la imagen con nuevas dimensiones 224x224
    # Luego pasamos la imagen modificada en tamaño al preprocesamiento de nuestra red. La red
    # que utilizaremos de ejemplo tiene por nombre Xception
    final_image = tf.keras.applications.xception.preprocess_input(resized_image)
    # Finalmente se retorna una imagen pre procesada (según lo indique el preprocess de xception)
    # y su etiqueta
    return final_image, label

In [6]:
# En este apartado hacemos los batch de datos directamente desde el dataset cargado por tensorflow
batch_size = 32

# Mezclamos el dataset 
train_set = train_set.shuffle(1000)
# Tanto para training, test y validación, aplicamos la función de preprocesamiento (preprocess)
# y luego generamos los batch de datos
train_set = train_set.map(preprocess).batch(batch_size).prefetch(1)
test_set = test_set.map(preprocess).batch(batch_size).prefetch(1)
valid_set = valid_set.map(preprocess).batch(batch_size).prefetch(1)

In [7]:
# Acá empezamos con transfer learning. Lo que haremos será cargar una arquitectura de red neuronal desde
# tensorflow ya entrenada. Eso quiere decir que, no solo estamos cargando la arquitectura, con sus neuronas
# y conexiones, sino que también estamos cargando los PESOS DE ENTRENAMIENTO.

# Weights indica si utilizaremos pesos pre entrenados con el dataset imagenet o no
# include_top es el parámetro que indica explícitamente si quieres o no la capa de salida original
# de esta red.
# En base_model tenemos cargado nuestro modelo Xception sin la capa de salida. Nosotros podemos
# poner NUESTRA PROPIA CAPA(S) DE SALIDA
base_model = tf.keras.applications.xception.Xception(weights="imagenet", include_top=False)
# Acá agregamos nuestras capas adicionales
avg = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
output = tf.keras.layers.Dense(5, activation="softmax")(avg)

# Podemos conectar el input de nuestro modelo base con el output recién generado a través de 
# un modelo de Keras
model = tf.keras.Model(inputs=base_model.input, outputs=output)
# En este punto tenemos un nuevo modelo que utiliza como base la arquitectura Xception

In [8]:
# También nosotros conversamos que es posible decidir si queremos reentrenar aquellas capas ya entrenadas

# Acá vamos capa por capa modificando el parámetro "trainable" que permite (o no) reentrenar dicha capa
for layer in base_model.layers:
    layer.trainable = False # Esto impide que las capas se re entrenen

# Con este for, sobre todas las capas de nuestro modelo, estamos impidiendo que se reentrene
# alguna de sus capas ya entrenadas

In [9]:
# Proceso de compilación  (tal cual vimos en las clases anteriores)
model.compile(loss="sparse_categorical_crossentropy", optimizer="sgd", 
              metrics=["accuracy"])

In [10]:
history = model.fit(train_set, epochs=10, validation_data=valid_set)

Epoch 1/10


2023-10-23 04:49:50.311056: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [2]
	 [[{{node Placeholder/_0}}]]
2023-10-23 04:49:50.311542: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_1' with dtype string and shape [2]
	 [[{{node Placeholder/_1}}]]


86/86 [==============================] - ETA: 0s - loss: 0.9824 - accuracy: 0.6842

2023-10-23 04:51:07.906730: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]
2023-10-23 04:51:07.910169: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_2' with dtype int64 and shape [1]
	 [[{{node Placeholder/_2}}]]


86/86 [==============================] - 99s 1s/step - loss: 0.9824 - accuracy: 0.6842 - val_loss: 0.7195 - val_accuracy: 0.7713
Epoch 2/10
86/86 [==============================] - 91s 1s/step - loss: 0.6005 - accuracy: 0.8256 - val_loss: 0.6019 - val_accuracy: 0.8040
Epoch 3/10
86/86 [==============================] - 99s 1s/step - loss: 0.5050 - accuracy: 0.8463 - val_loss: 0.5443 - val_accuracy: 0.8022
Epoch 4/10
86/86 [==============================] - 98s 1s/step - loss: 0.4541 - accuracy: 0.8601 - val_loss: 0.5100 - val_accuracy: 0.8240
Epoch 5/10
86/86 [==============================] - 97s 1s/step - loss: 0.4209 - accuracy: 0.8670 - val_loss: 0.4860 - val_accuracy: 0.8385
Epoch 6/10
86/86 [==============================] - 99s 1s/step - loss: 0.3960 - accuracy: 0.8739 - val_loss: 0.4715 - val_accuracy: 0.8512
Epoch 7/10
86/86 [==============================] - 94s 1s/step - loss: 0.3760 - accuracy: 0.8779 - val_loss: 0.4561 - val_accuracy: 0.8439
Epoch 8/10
86/86 [=============

In [11]:
model.predict(test_set)

2023-10-23 05:05:54.341007: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_1' with dtype string and shape [1]
	 [[{{node Placeholder/_1}}]]
2023-10-23 05:05:54.341250: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]


12/12 [==============================] - 11s 864ms/step


array([[6.6685770e-04, 2.9892201e-04, 9.8979938e-01, 2.8097844e-03,
        6.4251730e-03],
       [6.0550570e-02, 1.3898717e-01, 3.8048562e-01, 3.8398203e-01,
        3.5994679e-02],
       [2.6642814e-01, 2.5243647e-02, 6.7721091e-02, 6.2200689e-01,
        1.8600171e-02],
       ...,
       [9.9802881e-01, 5.0809089e-04, 5.8654271e-04, 6.1489292e-04,
        2.6166797e-04],
       [4.7372892e-01, 1.0460367e-01, 7.6526240e-02, 2.4113403e-01,
        1.0400708e-01],
       [1.4060043e-01, 2.6324868e-02, 1.0195327e-02, 8.0650765e-01,
        1.6371686e-02]], dtype=float32)

In [15]:
import numpy as np
for image, label in test_set:
    print("Etiqueta real",label)
    print("Predicción", model.predict(image))


Etiqueta real tf.Tensor([2 3 3 4 3 0 0 0 0 1 3 2 4 1 2 1 2 4 2 2 0 0 0 2 0 3 0 1 1 1 2 1], shape=(32,), dtype=int64)
1/1 [==============================] - 1s 781ms/step
Predicción [[6.66857697e-04 2.98922008e-04 9.89799380e-01 2.80978438e-03
  6.42517302e-03]
 [6.05505705e-02 1.38987169e-01 3.80485624e-01 3.83982033e-01
  3.59946787e-02]
 [2.66428143e-01 2.52436474e-02 6.77210912e-02 6.22006893e-01
  1.86001714e-02]
 [2.16502219e-01 7.42468759e-02 4.62541848e-01 1.43394426e-01
  1.03314690e-01]
 [5.50437830e-02 1.58021098e-03 3.97246424e-03 9.38655794e-01
  7.47800397e-04]
 [8.50617886e-01 8.73946473e-02 1.48381861e-02 4.66650501e-02
  4.84286895e-04]
 [8.62499416e-01 7.13406205e-02 2.73214392e-02 3.35697420e-02
  5.26876422e-03]
 [9.69361603e-01 6.69054314e-03 4.92229639e-03 1.12805264e-02
  7.74502940e-03]
 [9.80928719e-01 4.57983837e-03 7.78052537e-03 5.77170216e-03
  9.39211110e-04]
 [7.30425045e-02 7.59465873e-01 7.70783499e-02 8.47055092e-02
  5.70774777e-03]
 [2.18166739e-01 6.

1/1 [==============================] - 1s 791ms/step
Predicción [[1.28935243e-03 9.83915627e-01 3.34924698e-04 1.40727321e-02
  3.87397624e-04]
 [1.41795317e-04 1.02971731e-04 2.76326090e-02 2.80987658e-02
  9.44023848e-01]
 [9.31945264e-01 4.81927842e-02 3.44734266e-03 1.51363648e-02
  1.27823115e-03]
 [6.58326130e-03 1.35624083e-02 8.85291025e-02 2.19757650e-02
  8.69349480e-01]
 [1.19011886e-02 6.27228152e-03 7.17568323e-02 4.06908616e-03
  9.06000733e-01]
 [1.28727928e-02 1.93340834e-02 4.28641848e-02 1.39740035e-01
  7.85188913e-01]
 [7.91285202e-05 4.00638179e-04 6.73750937e-01 8.82677175e-03
  3.16942573e-01]
 [9.93540347e-01 8.24956282e-04 1.96920033e-03 2.91799637e-03
  7.47605984e-04]
 [4.53580637e-03 1.56738702e-02 2.41671391e-02 3.43329683e-02
  9.21290219e-01]
 [2.89185941e-01 1.42022207e-01 4.24210072e-01 1.16699100e-01
  2.78826728e-02]
 [2.95697879e-02 2.04550773e-02 5.42525277e-02 8.75943899e-01
  1.97787639e-02]
 [7.91339378e-04 6.02900051e-04 1.50577441e-01 1.3076209

1/1 [==============================] - 1s 791ms/step
Predicción [[5.97544189e-04 9.98435795e-01 1.11400012e-04 7.91038212e-04
  6.41981387e-05]
 [8.24785590e-01 1.19063102e-01 5.65227866e-03 4.95329797e-02
  9.66151129e-04]
 [4.90651615e-02 1.07584754e-03 2.64155841e-03 9.40135300e-01
  7.08207209e-03]
 [8.00983887e-03 6.91076135e-03 7.67649412e-01 9.07512847e-03
  2.08354861e-01]
 [5.74924634e-04 9.98185575e-01 9.34902055e-05 1.07064191e-03
  7.52956985e-05]
 [1.79369736e-03 8.88752809e-04 4.80574876e-01 1.08594857e-02
  5.05883276e-01]
 [3.21652330e-02 1.82315130e-02 5.18725634e-01 3.31083477e-01
  9.97941718e-02]
 [3.17412266e-03 9.85830367e-01 4.70894534e-04 1.02125881e-02
  3.12050106e-04]
 [3.58890593e-02 5.09013934e-03 1.24713837e-03 9.56552982e-01
  1.22060918e-03]
 [9.88507748e-01 1.44755119e-03 2.17429479e-03 6.28592027e-03
  1.58454478e-03]
 [9.70340908e-01 4.79939859e-03 3.94285796e-03 1.98596399e-02
  1.05716079e-03]
 [2.39554029e-02 5.09261852e-03 9.11547601e-01 4.9798241

1/1 [==============================] - 0s 451ms/step
Predicción [[9.90770936e-01 7.91054568e-04 8.28691642e-04 7.21496483e-03
  3.94398987e-04]
 [5.09495616e-01 1.04108304e-01 1.22706108e-01 1.47032544e-01
  1.16657458e-01]
 [5.69662787e-02 4.04489823e-02 8.33261192e-01 5.68997040e-02
  1.24238338e-02]
 [2.52688546e-02 6.05482087e-02 2.73746043e-01 7.12845474e-02
  5.69152415e-01]
 [4.15862761e-02 2.04539765e-03 6.88098185e-03 9.33386505e-01
  1.61008164e-02]
 [7.55683184e-01 8.96248370e-02 8.64248946e-02 6.53093383e-02
  2.95773731e-03]
 [1.24209292e-01 7.40037143e-01 8.10684487e-02 3.53667401e-02
  1.93184223e-02]
 [2.53827870e-01 2.39672929e-01 1.93530843e-01 2.40665898e-01
  7.23024383e-02]
 [1.31188836e-02 3.22738439e-02 1.72454134e-01 1.48215545e-02
  7.67331600e-01]
 [9.64861810e-01 2.84304153e-02 1.80679676e-03 3.94837931e-03
  9.52618604e-04]
 [1.05066476e-02 9.45060670e-01 8.59547034e-03 2.98420247e-02
  5.99512877e-03]
 [5.58023825e-02 7.14516118e-02 5.59280157e-01 2.8676748